In [1]:
import numpy as np
import numpy.testing as npt

from scipy import stats
from scipy.stats import t,ttest_ind
from scipy.stats import f
from scipy.stats import f_oneway
from scipy.stats import pearsonr

import statsmodels.api as sm
from statsmodels.regression._prediction import get_prediction
from statsmodels.stats.outliers_influence import OLSInfluence,MLEInfluence
from statsmodels.graphics.gofplots import qqplot_2samples,ProbPlot,qqplot
import pandas as pd
from patsy import dmatrices
from numpy.testing import assert_almost_equal, assert_allclose
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns

import os

# some_file.py
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, r'C:\Users\TODO\Desktop\Abhi\AI\AI\Math\Hands-On\statemodelsStudy')
from olsRegressionAnalysis import dispAnalysisOfVariance, tableDispFormatt,getInvOfProductMat,\
                                  getRegressionEqn,\
                                  dispReghressionAnalysis,norm_scalling,getCorrelation,\
                                  get_variance_inflation_factors

In [2]:
path = os.path.join(os.getcwd(), 'DataSet', 'TABLE_3_2_DeliveryTimeData.csv')
df = pd.read_csv(path)
df.columns = ['Obs','DlvrTImeY','NumCaseX1','DstX2']

print(df.columns)

Index(['Obs', 'DlvrTImeY', 'NumCaseX1', 'DstX2'], dtype='object')


In [3]:
y, X = dmatrices(
                 formula_like = 'DlvrTImeY ~ NumCaseX1 + DstX2', 
                 data=df,
                 return_type='dataframe'
                 )
res = sm.OLS(y, X).fit()


Matrix product contions

A mat MxN

B mat NxP

C = AB of mat MxP


In [4]:
def isMatProductConitionTrus(A,B):
    colA = A.shape[1]
    rowB = B.shape[0]
    if colA == rowB:
        return True
    else:
        return False

In [5]:
def getCIOfNewObs(procutMat,newObs,oslResult,alfa):
    if isMatProductConitionTrus(procutMat,newObs) == False:
        raise Exception("Shape of Mat8iirix A: ",procutMat.shape, "Shape of Matrix B: ",newObs.shape, 'are not Equal')    
    ret = np.matmul(np.matmul(newObs.getT(),procutMat),newObs)
    tSig = t.isf(q = alfa, df = oslResult.df_resid, loc=0, scale=1)
    predictOnNewObs = np.matmul(oslResult.params,newObs)
    ret = np.sqrt((1+ret.item()) * oslResult.mse_resid)*tSig
    ci_u = predictOnNewObs + ret
    ci_l = predictOnNewObs - ret
    CI = np.column_stack((ci_l, ci_u))
    print(CI)
    return CI


In [6]:
# Please apply formula PG 193

x = np.mat(np.array([1,8,275]).reshape(3,1))
getCIOfNewObs(
               procutMat = getInvOfProductMat(
                                                formula = 'DlvrTImeY ~ NumCaseX1 + DstX2',
                                                df_data = df
                                                ),
               newObs    = x,
               oslResult = res,
               alfa = 0.05/2
              )


<class 'numpy.matrix'> (25, 3)
[[12.28455895 26.16407315]]


array([[12.28455895, 26.16407315]])

# Plot of CI

TODO